# Triton Inference Server 教程 (Triton Tutorial)

> **前置知识**: ONNX 模型格式、REST API 基础、Docker 基础
>
> **学习目标**: 掌握 Triton Inference Server 的配置和使用

---

## 什么是 Triton Inference Server？

```
┌─────────────────────────────────────────────────────────────┐
│                  Triton Inference Server                     │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌─────────┐  ┌─────────┐  ┌─────────┐  ┌─────────┐        │
│  │ Model A │  │ Model B │  │ Model C │  │ Model D │        │
│  │ (ONNX)  │  │ (TRT)   │  │(PyTorch)│  │  (TF)   │        │
│  └─────────┘  └─────────┘  └─────────┘  └─────────┘        │
│                                                             │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  核心功能:                                          │   │
│  │  - 动态批处理: 自动收集请求批量处理                │   │
│  │  - 模型并发: 多模型同时服务                        │   │
│  │  - 版本管理: 支持模型热更新                        │   │
│  │  - GPU 调度: 智能分配 GPU 资源                     │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  接口: HTTP/REST (8000) │ gRPC (8001) │ Metrics (8002)│ │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

## 本教程内容

1. **数据类型映射** - Triton 与 NumPy 类型转换
2. **模型仓库结构** - 标准化目录组织
3. **模型配置生成** - config.pbtxt 配置文件
4. **Ensemble 模型** - 模型流水线
5. **客户端使用** - HTTP 和 gRPC 客户端
6. **性能优化** - 动态批处理配置

In [ ]:
# ============================================================
# 环境准备
# ============================================================
import numpy as np
from enum import Enum
from typing import List, Dict, Any, Optional
from dataclasses import dataclass

# 设置随机种子
np.random.seed(42)

print("=" * 60)
print("环境准备完成")
print("=" * 60)

# 检查 Triton 客户端
try:
    import tritonclient.http as httpclient
    TRITON_HTTP_AVAILABLE = True
    print(f"\n✓ Triton HTTP 客户端已安装")
except ImportError:
    TRITON_HTTP_AVAILABLE = False
    print(f"\n✗ Triton HTTP 客户端未安装")

try:
    import tritonclient.grpc as grpcclient
    TRITON_GRPC_AVAILABLE = True
    print(f"✓ Triton gRPC 客户端已安装")
except ImportError:
    TRITON_GRPC_AVAILABLE = False
    print(f"✗ Triton gRPC 客户端未安装")

TRITON_AVAILABLE = TRITON_HTTP_AVAILABLE or TRITON_GRPC_AVAILABLE

if not TRITON_AVAILABLE:
    print(f"\n安装命令: pip install tritonclient[all]")

print(f"\n注意: 本教程的配置生成功能不依赖 Triton 客户端")
print(f"      可以独立学习模型配置和仓库结构")

## 1. Triton 数据类型

**核心概念**: Triton 使用特定的数据类型，需要与 NumPy 类型进行转换

```
┌─────────────────────────────────────────────────────────────┐
│                    数据类型映射                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  NumPy 类型              Triton 类型                        │
│  ─────────────           ───────────                        │
│  np.float32       →      TYPE_FP32                         │
│  np.float64       →      TYPE_FP64                         │
│  np.float16       →      TYPE_FP16                         │
│  np.int32         →      TYPE_INT32                        │
│  np.int64         →      TYPE_INT64                        │
│  np.uint8         →      TYPE_UINT8                        │
│  np.bool_         →      TYPE_BOOL                         │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# Triton 数据类型定义 (自包含实现)
# ============================================================
print("=" * 60)
print("Triton 数据类型")
print("=" * 60)

class TritonDataType(Enum):
    """Triton 支持的数据类型"""
    TYPE_BOOL = "BOOL"
    TYPE_UINT8 = "UINT8"
    TYPE_UINT16 = "UINT16"
    TYPE_UINT32 = "UINT32"
    TYPE_UINT64 = "UINT64"
    TYPE_INT8 = "INT8"
    TYPE_INT16 = "INT16"
    TYPE_INT32 = "INT32"
    TYPE_INT64 = "INT64"
    TYPE_FP16 = "FP16"
    TYPE_FP32 = "FP32"
    TYPE_FP64 = "FP64"
    TYPE_STRING = "BYTES"


# NumPy 到 Triton 类型映射
NUMPY_TO_TRITON_DTYPE = {
    np.dtype('bool'): "TYPE_BOOL",
    np.dtype('uint8'): "TYPE_UINT8",
    np.dtype('uint16'): "TYPE_UINT16",
    np.dtype('uint32'): "TYPE_UINT32",
    np.dtype('uint64'): "TYPE_UINT64",
    np.dtype('int8'): "TYPE_INT8",
    np.dtype('int16'): "TYPE_INT16",
    np.dtype('int32'): "TYPE_INT32",
    np.dtype('int64'): "TYPE_INT64",
    np.dtype('float16'): "TYPE_FP16",
    np.dtype('float32'): "TYPE_FP32",
    np.dtype('float64'): "TYPE_FP64",
}


def numpy_to_triton_dtype(np_dtype) -> str:
    """将 NumPy 数据类型转换为 Triton 数据类型"""
    return NUMPY_TO_TRITON_DTYPE.get(np.dtype(np_dtype), "TYPE_FP32")


print("\nTriton 数据类型:")
for dtype in TritonDataType:
    print(f"  {dtype.name}: {dtype.value}")

In [ ]:
# ============================================================
# NumPy 到 Triton 类型转换测试
# ============================================================
print("=" * 60)
print("NumPy → Triton 类型转换")
print("=" * 60)

test_arrays = [
    np.array([1.0], dtype=np.float32),
    np.array([1.0], dtype=np.float64),
    np.array([1.0], dtype=np.float16),
    np.array([1], dtype=np.int32),
    np.array([1], dtype=np.int64),
    np.array([1], dtype=np.uint8),
    np.array([True], dtype=np.bool_),
]

print(f"\n{'NumPy 类型':<20} {'Triton 类型':<15}")
print("-" * 35)
for arr in test_arrays:
    triton_dtype = numpy_to_triton_dtype(arr.dtype)
    print(f"{str(arr.dtype):<20} {triton_dtype:<15}")

## 2. 模型仓库结构

**核心概念**: Triton 使用特定的目录结构来组织模型

```
┌─────────────────────────────────────────────────────────────┐
│                    模型仓库结构                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  model_repository/                                          │
│  ├── model_a/                    # 模型目录                 │
│  │   ├── config.pbtxt            # 模型配置文件             │
│  │   ├── 1/                      # 版本 1                   │
│  │   │   └── model.onnx          # ONNX 模型文件            │
│  │   └── 2/                      # 版本 2                   │
│  │       └── model.onnx          # 新版本模型               │
│  │                                                          │
│  ├── model_b/                                               │
│  │   ├── config.pbtxt                                       │
│  │   └── 1/                                                 │
│  │       └── model.plan          # TensorRT 引擎            │
│  │                                                          │
│  └── ensemble_model/             # 模型流水线               │
│      ├── config.pbtxt                                       │
│      └── 1/                                                 │
│          └── (empty)             # Ensemble 无模型文件      │
│                                                             │
│  支持的模型格式:                                            │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  .onnx  - ONNX Runtime 后端                        │   │
│  │  .plan  - TensorRT 后端                            │   │
│  │  .pt    - PyTorch 后端                             │   │
│  │  .savedmodel - TensorFlow 后端                     │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

## 3. 模型配置生成

**核心概念**: config.pbtxt 是 Triton 的模型配置文件，定义输入输出、批处理等参数

```
┌─────────────────────────────────────────────────────────────┐
│                    config.pbtxt 核心配置                     │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  基本配置:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  name: 模型名称                                     │   │
│  │  platform: 后端类型 (onnxruntime_onnx, tensorrt_plan)│  │
│  │  max_batch_size: 最大批次大小                       │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  输入输出配置:                                              │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  input/output:                                      │   │
│  │    - name: 张量名称                                 │   │
│  │    - data_type: 数据类型                            │   │
│  │    - dims: 维度 (不含 batch)                        │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  动态批处理:                                                │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  dynamic_batching:                                  │   │
│  │    - preferred_batch_size: 首选批次大小             │   │
│  │    - max_queue_delay_microseconds: 最大等待时间     │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 模型配置生成器 (自包含实现)
# ============================================================
print("=" * 60)
print("模型配置生成器")
print("=" * 60)

class ModelConfigGenerator:
    """
    Triton 模型配置生成器
    
    生成 config.pbtxt 配置文件内容
    """
    
    @staticmethod
    def generate_config(
        name: str,
        platform: str,
        max_batch_size: int,
        inputs: List[Dict],
        outputs: List[Dict],
        dynamic_batching: bool = True,
        preferred_batch_sizes: List[int] = None,
        max_queue_delay_microseconds: int = 100,
        instance_count: int = 1,
        device: str = "GPU"
    ) -> str:
        """
        生成模型配置
        
        参数:
            name: 模型名称
            platform: 后端平台 (onnxruntime_onnx, tensorrt_plan, pytorch_libtorch)
            max_batch_size: 最大批次大小
            inputs: 输入配置列表
            outputs: 输出配置列表
            dynamic_batching: 是否启用动态批处理
            preferred_batch_sizes: 首选批次大小列表
            max_queue_delay_microseconds: 最大队列等待时间
            instance_count: 模型实例数
            device: 设备类型 (GPU/CPU)
        """
        config = f'name: "{name}"\n'
        config += f'platform: "{platform}"\n'
        config += f'max_batch_size: {max_batch_size}\n\n'
        
        # 输入配置
        for inp in inputs:
            config += 'input [\n'
            config += '  {\n'
            config += f'    name: "{inp["name"]}"\n'
            config += f'    data_type: {inp["data_type"]}\n'
            config += f'    dims: {inp["dims"]}\n'
            config += '  }\n'
            config += ']\n\n'
        
        # 输出配置
        for out in outputs:
            config += 'output [\n'
            config += '  {\n'
            config += f'    name: "{out["name"]}"\n'
            config += f'    data_type: {out["data_type"]}\n'
            config += f'    dims: {out["dims"]}\n'
            config += '  }\n'
            config += ']\n\n'
        
        # 动态批处理配置
        if dynamic_batching:
            config += 'dynamic_batching {\n'
            if preferred_batch_sizes:
                config += f'  preferred_batch_size: {preferred_batch_sizes}\n'
            config += f'  max_queue_delay_microseconds: {max_queue_delay_microseconds}\n'
            config += '}\n\n'
        
        # 实例配置
        kind = "KIND_GPU" if device.upper() == "GPU" else "KIND_CPU"
        config += 'instance_group [\n'
        config += '  {\n'
        config += f'    count: {instance_count}\n'
        config += f'    kind: {kind}\n'
        if device.upper() == "GPU":
            config += '    gpus: [ 0 ]\n'
        config += '  }\n'
        config += ']\n'
        
        return config
    
    @staticmethod
    def generate_ensemble_config(
        name: str,
        max_batch_size: int,
        inputs: List[Dict],
        outputs: List[Dict],
        steps: List[Dict]
    ) -> str:
        """生成 Ensemble 模型配置"""
        config = f'name: "{name}"\n'
        config += 'platform: "ensemble"\n'
        config += f'max_batch_size: {max_batch_size}\n\n'
        
        # 输入配置
        for inp in inputs:
            config += 'input [\n'
            config += '  {\n'
            config += f'    name: "{inp["name"]}"\n'
            config += f'    data_type: {inp["data_type"]}\n'
            config += f'    dims: {inp["dims"]}\n'
            config += '  }\n'
            config += ']\n\n'
        
        # 输出配置
        for out in outputs:
            config += 'output [\n'
            config += '  {\n'
            config += f'    name: "{out["name"]}"\n'
            config += f'    data_type: {out["data_type"]}\n'
            config += f'    dims: {out["dims"]}\n'
            config += '  }\n'
            config += ']\n\n'
        
        # Ensemble 调度配置
        config += 'ensemble_scheduling {\n'
        for step in steps:
            config += '  step [\n'
            config += '    {\n'
            config += f'      model_name: "{step["model_name"]}"\n'
            config += f'      model_version: {step["model_version"]}\n'
            for key, value in step.get("input_map", {}).items():
                config += f'      input_map {{ key: "{key}" value: "{value}" }}\n'
            for key, value in step.get("output_map", {}).items():
                config += f'      output_map {{ key: "{key}" value: "{value}" }}\n'
            config += '    }\n'
            config += '  ]\n'
        config += '}\n'
        
        return config


# 生成基本配置示例
config = ModelConfigGenerator.generate_config(
    name="image_classifier",
    platform="onnxruntime_onnx",
    max_batch_size=32,
    inputs=[{
        "name": "input",
        "data_type": "TYPE_FP32",
        "dims": [3, 224, 224]
    }],
    outputs=[{
        "name": "output",
        "data_type": "TYPE_FP32",
        "dims": [1000]
    }],
    dynamic_batching=True,
    preferred_batch_sizes=[4, 8, 16, 32],
    instance_count=2,
    device="GPU"
)

print("\n生成的 config.pbtxt:")
print("=" * 50)
print(config)

In [ ]:
# ============================================================
# CPU 模型配置示例
# ============================================================
print("=" * 60)
print("CPU 模型配置")
print("=" * 60)

cpu_config = ModelConfigGenerator.generate_config(
    name="text_encoder",
    platform="onnxruntime_onnx",
    max_batch_size=16,
    inputs=[{
        "name": "input_ids",
        "data_type": "TYPE_INT64",
        "dims": [512]
    }],
    outputs=[{
        "name": "embeddings",
        "data_type": "TYPE_FP32",
        "dims": [768]
    }],
    device="CPU",
    instance_count=4,
    dynamic_batching=True,
    preferred_batch_sizes=[4, 8, 16]
)

print("\nCPU 模型配置 (适用于 NLP 模型):")
print("=" * 50)
print(cpu_config)

print("\nCPU vs GPU 配置差异:")
print("  - kind: KIND_CPU (无需指定 gpus)")
print("  - instance_count: 通常设置更多实例 (CPU 并行)")
print("  - 适用场景: 文本处理、小模型、无 GPU 环境")

<cell_type>markdown</cell_type>## 4. Ensemble 模型配置

**核心概念**: Ensemble 模型将多个模型串联成流水线，实现复杂推理任务

```
┌─────────────────────────────────────────────────────────────┐
│                    Ensemble 模型流水线                       │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  输入: raw_image                                            │
│       │                                                     │
│       ▼                                                     │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  Step 1: preprocessing                              │   │
│  │  - 输入: raw_input ← raw_image                     │   │
│  │  - 输出: processed_output → preprocessed           │   │
│  │  - 功能: 图像预处理 (resize, normalize)            │   │
│  └─────────────────────────────────────────────────────┘   │
│       │                                                     │
│       ▼ preprocessed                                        │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  Step 2: classifier                                 │   │
│  │  - 输入: input ← preprocessed                      │   │
│  │  - 输出: output → classification                   │   │
│  │  - 功能: 图像分类                                  │   │
│  └─────────────────────────────────────────────────────┘   │
│       │                                                     │
│       ▼                                                     │
│  输出: classification                                       │
│                                                             │
│  优势:                                                      │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  1. 模块化: 每个步骤独立开发和优化                 │   │
│  │  2. 复用: 预处理模型可被多个分类器共享             │   │
│  │  3. 灵活: 可动态替换流水线中的任意模型             │   │
│  │  4. 高效: Triton 自动优化数据传输                  │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# Ensemble 配置生成
# ============================================================
print("=" * 60)
print("Ensemble 模型配置")
print("=" * 60)

ensemble_config = ModelConfigGenerator.generate_ensemble_config(
    name="image_pipeline",
    max_batch_size=32,
    inputs=[{
        "name": "raw_image",
        "data_type": "TYPE_UINT8",
        "dims": [-1, -1, 3]  # 动态尺寸: [height, width, channels]
    }],
    outputs=[{
        "name": "classification",
        "data_type": "TYPE_FP32",
        "dims": [1000]
    }],
    steps=[
        {
            "model_name": "preprocessing",
            "model_version": -1,  # -1 表示使用最新版本
            "input_map": {"raw_input": "raw_image"},
            "output_map": {"processed_output": "preprocessed"}
        },
        {
            "model_name": "classifier",
            "model_version": -1,
            "input_map": {"input": "preprocessed"},
            "output_map": {"output": "classification"}
        }
    ]
)

print("\nEnsemble 配置 (图像分类流水线):")
print("=" * 50)
print(ensemble_config)

print("\n配置说明:")
print("  - platform: 'ensemble' (固定值)")
print("  - model_version: -1 表示使用最新版本")
print("  - input_map: 将 Ensemble 输入映射到子模型输入")
print("  - output_map: 将子模型输出映射到 Ensemble 输出或下一步输入")

<cell_type>markdown</cell_type>## 5. 模型元数据

**核心概念**: 模型元数据描述模型的基本信息，用于客户端查询和验证

```
┌─────────────────────────────────────────────────────────────┐
│                    模型元数据结构                            │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ModelMetadata:                                             │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  name: "resnet50"                                   │   │
│  │  versions: ["1", "2", "3"]                          │   │
│  │  platform: "onnxruntime_onnx"                       │   │
│  │                                                     │   │
│  │  inputs:                                            │   │
│  │    - name: "input"                                  │   │
│  │      datatype: "FP32"                               │   │
│  │      shape: [1, 3, 224, 224]                        │   │
│  │                                                     │   │
│  │  outputs:                                           │   │
│  │    - name: "output"                                 │   │
│  │      datatype: "FP32"                               │   │
│  │      shape: [1, 1000]                               │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  用途:                                                      │
│  - 客户端验证输入输出格式                                  │
│  - 自动生成客户端代码                                      │
│  - 模型版本管理                                            │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 模型元数据定义 (自包含实现)
# ============================================================
print("=" * 60)
print("模型元数据")
print("=" * 60)

@dataclass
class ModelInput:
    """模型输入描述"""
    name: str
    datatype: str
    shape: List[int]

@dataclass
class ModelOutput:
    """模型输出描述"""
    name: str
    datatype: str
    shape: List[int]

@dataclass
class ModelMetadata:
    """模型元数据"""
    name: str
    versions: List[str]
    platform: str
    inputs: List[ModelInput]
    outputs: List[ModelOutput]


# 创建模型元数据示例
metadata = ModelMetadata(
    name="resnet50",
    versions=["1", "2", "3"],
    platform="onnxruntime_onnx",
    inputs=[
        ModelInput(name="input", datatype="FP32", shape=[1, 3, 224, 224])
    ],
    outputs=[
        ModelOutput(name="output", datatype="FP32", shape=[1, 1000])
    ]
)

print("\n模型元数据:")
print(f"  名称: {metadata.name}")
print(f"  版本: {metadata.versions}")
print(f"  平台: {metadata.platform}")
print(f"\n输入:")
for inp in metadata.inputs:
    print(f"  - {inp.name}: {inp.datatype} {inp.shape}")
print(f"\n输出:")
for out in metadata.outputs:
    print(f"  - {out.name}: {out.datatype} {out.shape}")

<cell_type>markdown</cell_type>## 6. Triton 客户端使用

**核心概念**: Triton 提供 HTTP 和 gRPC 两种客户端协议

```
┌─────────────────────────────────────────────────────────────┐
│                    客户端协议对比                            │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  HTTP (端口 8000):                                          │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  优点: 简单易用，兼容性好，便于调试                 │   │
│  │  缺点: 延迟较高，序列化开销大                       │   │
│  │  适用: 开发测试，低频请求                           │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  gRPC (端口 8001):                                          │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  优点: 低延迟，高效二进制序列化，支持流式           │   │
│  │  缺点: 需要 protobuf，调试较复杂                    │   │
│  │  适用: 生产环境，高频请求                           │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  推荐: 生产环境使用 gRPC，开发测试使用 HTTP                │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

**注意**: 以下代码需要运行中的 Triton 服务器

In [ ]:
# ============================================================
# Triton 客户端使用示例
# ============================================================
print("=" * 60)
print("Triton 客户端使用")
print("=" * 60)

print("""
# ============================================================
# HTTP 客户端示例
# ============================================================
import tritonclient.http as httpclient
import numpy as np

# 创建客户端
client = httpclient.InferenceServerClient(url="localhost:8000")

# 检查服务器状态
if client.is_server_live():
    print("服务器运行中")

if client.is_server_ready():
    print("服务器就绪")

# 检查模型状态
if client.is_model_ready("resnet50"):
    print("模型就绪")

# 获取模型元数据
metadata = client.get_model_metadata("resnet50")
print(f"模型输入: {metadata['inputs']}")
print(f"模型输出: {metadata['outputs']}")

# 准备输入数据
input_data = np.random.randn(1, 3, 224, 224).astype(np.float32)

# 创建输入对象
inputs = [
    httpclient.InferInput("input", input_data.shape, "FP32")
]
inputs[0].set_data_from_numpy(input_data)

# 执行推理
result = client.infer(model_name="resnet50", inputs=inputs)

# 获取输出
output = result.as_numpy("output")
print(f"输出形状: {output.shape}")

# ============================================================
# gRPC 客户端示例 (更高性能)
# ============================================================
import tritonclient.grpc as grpcclient

# 创建 gRPC 客户端
grpc_client = grpcclient.InferenceServerClient(url="localhost:8001")

# 创建输入
grpc_inputs = [
    grpcclient.InferInput("input", input_data.shape, "FP32")
]
grpc_inputs[0].set_data_from_numpy(input_data)

# 执行推理
grpc_result = grpc_client.infer(model_name="resnet50", inputs=grpc_inputs)
grpc_output = grpc_result.as_numpy("output")
""")

if TRITON_AVAILABLE:
    print("\n✓ Triton 客户端已安装，可以运行上述代码")
else:
    print("\n✗ Triton 客户端未安装")
    print("  安装命令: pip install tritonclient[all]")

<cell_type>markdown</cell_type>## 7. 批量推理

**核心概念**: 批量推理可以显著提高吞吐量，Triton 支持动态批处理

```
┌─────────────────────────────────────────────────────────────┐
│                    批量推理流程                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  客户端发送多个请求:                                        │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  请求 1: input_data_1                               │   │
│  │  请求 2: input_data_2                               │   │
│  │  请求 3: input_data_3                               │   │
│  │  ...                                                │   │
│  └─────────────────────────────────────────────────────┘   │
│                        │                                    │
│                        ▼                                    │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  Triton 动态批处理:                                 │   │
│  │  - 收集多个请求                                     │   │
│  │  - 组合成批次 [batch_size, ...]                    │   │
│  │  - 一次 GPU 推理                                    │   │
│  │  - 分发结果到各请求                                 │   │
│  └─────────────────────────────────────────────────────┘   │
│                        │                                    │
│                        ▼                                    │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  结果 1: output_1                                   │   │
│  │  结果 2: output_2                                   │   │
│  │  结果 3: output_3                                   │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 批量推理示例代码
# ============================================================
print("=" * 60)
print("批量推理示例")
print("=" * 60)

print("""
# ============================================================
# 方法 1: 单次批量请求 (推荐)
# ============================================================
import tritonclient.http as httpclient
import numpy as np

client = httpclient.InferenceServerClient(url="localhost:8000")

# 准备批量输入 (batch_size=4)
batch_data = np.random.randn(4, 3, 224, 224).astype(np.float32)

inputs = [
    httpclient.InferInput("input", batch_data.shape, "FP32")
]
inputs[0].set_data_from_numpy(batch_data)

# 单次推理处理整个批次
result = client.infer(model_name="resnet50", inputs=inputs)
batch_output = result.as_numpy("output")  # shape: [4, 1000]

print(f"批量输出形状: {batch_output.shape}")

# ============================================================
# 方法 2: 异步并发请求
# ============================================================
import concurrent.futures

def single_infer(data):
    inputs = [httpclient.InferInput("input", data.shape, "FP32")]
    inputs[0].set_data_from_numpy(data)
    return client.infer(model_name="resnet50", inputs=inputs)

# 并发发送多个请求
data_list = [np.random.randn(1, 3, 224, 224).astype(np.float32) for _ in range(10)]

with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(single_infer, data) for data in data_list]
    results = [f.result() for f in futures]

print(f"完成 {len(results)} 个并发请求")
""")

print("\n批量推理优化建议:")
print("  1. 优先使用单次批量请求 (方法 1)")
print("  2. 批次大小应与 config.pbtxt 中的 preferred_batch_size 匹配")
print("  3. 异步并发适用于无法预先收集批次的场景")

<cell_type>markdown</cell_type>## 8. 启动 Triton 服务器

**核心概念**: 使用 Docker 启动 Triton 服务器是最简单的方式

```
┌─────────────────────────────────────────────────────────────┐
│                    Triton 服务器端口                         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  端口配置:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  8000: HTTP/REST API                                │   │
│  │  8001: gRPC API                                     │   │
│  │  8002: Prometheus Metrics                           │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  健康检查端点:                                              │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  GET /v2/health/live    - 服务器是否运行            │   │
│  │  GET /v2/health/ready   - 服务器是否就绪            │   │
│  │  GET /v2/models/{name}  - 模型状态                  │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 启动 Triton 服务器
# ============================================================
print("=" * 60)
print("启动 Triton 服务器")
print("=" * 60)

print("""
# ============================================================
# 使用 Docker 启动 (推荐)
# ============================================================

# 基本启动命令
docker run --gpus all --rm \\
    -p 8000:8000 -p 8001:8001 -p 8002:8002 \\
    -v /path/to/model_repository:/models \\
    nvcr.io/nvidia/tritonserver:23.10-py3 \\
    tritonserver --model-repository=/models

# ============================================================
# 常用参数说明
# ============================================================

# 模型控制模式
--model-control-mode=poll      # 自动检测模型更新
--model-control-mode=explicit  # 手动加载/卸载模型
--model-control-mode=none      # 启动时加载所有模型

# 轮询间隔 (配合 poll 模式)
--repository-poll-secs=30      # 每 30 秒检查模型更新

# 自动生成配置 (简化部署)
--strict-model-config=false    # 自动推断模型配置

# 日志级别
--log-verbose=1                # 详细日志

# 指定加载的模型
--load-model=model_a           # 只加载指定模型
--load-model=model_b

# ============================================================
# 验证服务器状态
# ============================================================

# 检查服务器健康状态
curl -v localhost:8000/v2/health/live
curl -v localhost:8000/v2/health/ready

# 获取模型列表
curl localhost:8000/v2/models

# 获取模型元数据
curl localhost:8000/v2/models/resnet50

# 获取 Prometheus 指标
curl localhost:8002/metrics
""")

print("\n启动检查清单:")
print("  ✓ 确保 model_repository 目录结构正确")
print("  ✓ 每个模型目录包含 config.pbtxt")
print("  ✓ 模型文件放在版本号目录下 (如 1/model.onnx)")
print("  ✓ 检查 GPU 驱动和 CUDA 版本兼容性")

<cell_type>markdown</cell_type>## 9. 性能优化配置

**核心概念**: 通过配置优化提高 Triton 推理性能

```
┌─────────────────────────────────────────────────────────────┐
│                    性能优化策略                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. 动态批处理优化:                                         │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  preferred_batch_size: [8, 16, 32, 64]             │   │
│  │  - 设置多个首选批次大小                             │   │
│  │  - Triton 会选择最接近的批次大小                    │   │
│  │                                                     │   │
│  │  max_queue_delay_microseconds: 50-100              │   │
│  │  - 较小值: 低延迟，可能批次较小                     │   │
│  │  - 较大值: 高吞吐，延迟略增                         │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  2. 实例配置:                                               │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  instance_count: 2-4                                │   │
│  │  - 多实例并行处理请求                               │   │
│  │  - 根据 GPU 内存和模型大小调整                      │   │
│  │                                                     │   │
│  │  kind: KIND_GPU / KIND_CPU                         │   │
│  │  - GPU 推理性能更高                                 │   │
│  │  - CPU 适用于小模型或无 GPU 环境                    │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  3. 数据类型优化:                                           │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  TYPE_FP16: 半精度，速度快，精度略降                │   │
│  │  TYPE_FP32: 单精度，精度高，速度较慢                │   │
│  │  TYPE_INT8: 量化模型，速度最快                      │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 高性能配置示例
# ============================================================
print("=" * 60)
print("高性能配置示例")
print("=" * 60)

high_perf_config = ModelConfigGenerator.generate_config(
    name="high_perf_model",
    platform="tensorrt_plan",  # TensorRT 后端性能最佳
    max_batch_size=64,
    inputs=[{
        "name": "input",
        "data_type": "TYPE_FP16",  # 使用 FP16 加速
        "dims": [3, 224, 224]
    }],
    outputs=[{
        "name": "output",
        "data_type": "TYPE_FP16",
        "dims": [1000]
    }],
    dynamic_batching=True,
    preferred_batch_sizes=[8, 16, 32, 64],  # 多个首选批次大小
    max_queue_delay_microseconds=50,  # 更短的等待时间
    instance_count=4,  # 更多实例
    device="GPU"
)

print("\n高性能配置 (TensorRT + FP16):")
print("=" * 50)
print(high_perf_config)

print("\n性能优化检查清单:")
print("  ✓ 使用 TensorRT 后端 (tensorrt_plan)")
print("  ✓ 启用 FP16 精度")
print("  ✓ 配置动态批处理")
print("  ✓ 设置多个首选批次大小")
print("  ✓ 配置多个模型实例")
print("  ✓ 调整队列等待时间")

print("\n预期性能提升:")
print("  - TensorRT vs ONNX: 2-5x 加速")
print("  - FP16 vs FP32: 1.5-2x 加速")
print("  - 动态批处理: 2-4x 吞吐量提升")

<cell_type>markdown</cell_type>## 总结

本教程介绍了 Triton Inference Server 的核心功能：

### 核心知识点

| 主题 | 关键内容 |
|:-----|:---------|
| 数据类型 | NumPy ↔ Triton 类型映射 |
| 模型仓库 | 标准化目录结构，版本管理 |
| config.pbtxt | 输入输出配置，动态批处理 |
| Ensemble | 模型流水线，input_map/output_map |
| 客户端 | HTTP (8000) / gRPC (8001) |
| 性能优化 | TensorRT、FP16、多实例 |

### API 速查

```python
# 模型配置生成
config = ModelConfigGenerator.generate_config(
    name="model_name",
    platform="onnxruntime_onnx",  # 或 tensorrt_plan
    max_batch_size=32,
    inputs=[{"name": "input", "data_type": "TYPE_FP32", "dims": [3, 224, 224]}],
    outputs=[{"name": "output", "data_type": "TYPE_FP32", "dims": [1000]}],
    dynamic_batching=True,
    preferred_batch_sizes=[4, 8, 16, 32]
)

# HTTP 客户端
import tritonclient.http as httpclient
client = httpclient.InferenceServerClient(url="localhost:8000")
inputs = [httpclient.InferInput("input", data.shape, "FP32")]
inputs[0].set_data_from_numpy(data)
result = client.infer(model_name="model", inputs=inputs)
output = result.as_numpy("output")

# Docker 启动
# docker run --gpus all -p 8000:8000 -p 8001:8001 -p 8002:8002 \
#     -v /path/to/models:/models nvcr.io/nvidia/tritonserver:23.10-py3 \
#     tritonserver --model-repository=/models
```

### 最佳实践

```
部署检查清单:
✓ 使用 TensorRT 后端获得最佳 GPU 性能
✓ 启用动态批处理提高吞吐量
✓ 使用 gRPC 协议减少延迟
✓ 配置多个模型实例实现并行推理
✓ 使用 Ensemble 构建复杂推理流水线
✓ 监控 Prometheus 指标 (端口 8002)

常见问题:
✗ 模型加载失败 → 检查 config.pbtxt 配置
✗ 推理超时 → 增加 max_queue_delay_microseconds
✗ 内存不足 → 减少 instance_count 或 max_batch_size
```

### 下一步学习

- **03_LoadBalancing_tutorial.ipynb**: 负载均衡与高可用
- **04_Advanced_Serving_tutorial.ipynb**: 高级服务技术